# 21 — Iterative Self-Improvement Loop
The model teaches itself ARO in rounds:

```
Round 0  →  base model (NB19 preference-SFT or NB18 SFT) generates data
            self-repair fixes failures with aro check
            fine-tune on passing samples
            fuse adapter → model_round_1

Round 1  →  model_round_1 generates NEW data (better quality)
            self-repair loop
            fine-tune on round_0 + round_1 data combined
            fuse → model_round_2

Round N  →  ...
```

Each round the model gets better at ARO → generates higher-quality data →
next fine-tune starts from a stronger point.

The initial dataset assembled by NB17 is included in round 0's training set,
so the loop builds directly on top of the warm-start, the SFT fine-tune, and
the preference-SFT pass.

**Improvements that mirror NB18/NB19** (added 2026-05-02):
- Pre-flight token filter (drops samples > MAX_SEQ_LEN before mlx_lm truncates)
- NaN guard (abort training immediately on NaN loss)
- Best-checkpoint selection (promote the lowest-val_loss adapter, not the final one)
- "No improvement vs best for N evals" early stop (replaces brittle "consecutive increases")
- `--steps-per-eval 50` (8 val checks per round instead of 2)
- Weight decay 0.01 via lora YAML config
- Smarter base-model selection (validates the candidate model before adopting it)
- Effective batch 16 (`BATCH_SIZE=2`, `--grad-accumulation-steps 8`)
- Bigger eval (`quick_eval(n=60)`) to halve variance
- Real weight fingerprint (samples bytes from start/middle/end of every shard)

**100% local, 100% open-source, runs on Apple Silicon.**

**Prerequisites:** notebooks 01–19 (or at minimum 01–04 + NB18 SFT)
**Install:** `pip install mlx-lm`

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'mlx-lm'], check=False)

In [ ]:
import json
import os
import re
import random
import subprocess
import sys
import importlib
import tempfile
import time
from collections import Counter
from pathlib import Path
from mlx_lm import load, generate as mlx_generate
from mlx_lm.sample_utils import make_sampler

try:
    SCRIPT_DIR = Path(__file__).parent.resolve()
except NameError:
    SCRIPT_DIR = Path('.').resolve()

if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))
import config; importlib.reload(config); from config import *

# Route all tempfile operations to the external volume — /var/folders fills up quickly
_TMP_ROOT = DATA_ROOT / 'tmp'
_TMP_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['TMPDIR'] = str(_TMP_ROOT)
tempfile.tempdir = str(_TMP_ROOT)   # also set Python-level override
print(f'Temp dir: {tempfile.gettempdir()}')

ITERATIVE_MODELS_DIR.mkdir(parents=True, exist_ok=True)
ROUNDS_DIR   = DATA_ROOT  / 'rounds'
ROUNDS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Data root:  {DATA_ROOT}')
print(f'Models dir: {ITERATIVE_MODELS_DIR}')
print(f'Rounds dir: {ROUNDS_DIR}')

## Loop config

In [ ]:
# Use MODEL_ID from config.py (respects TRAIN_ON_BASE flag).
# After each round this is replaced with the fused model path.
BASE_MODEL = MODEL_ID

# ── Smart base-model selection ─────────────────────────────────────────────
# Adopt a fine-tuned model only if it actually generates working output. This
# guards against the 2026-04-30 NB19 incident where a NaN'd DPO model was
# used as the iterative-loop baseline (pass rate: 0%).
#
# We do a fast 2-sample sanity check after the model loads (in cell 14 below).
# Here we just rank the candidates; the loop runner validates before adopting.
#
#   1. NB19 preference-SFT (highest quality if not failed)
#   2. NB18 SFT (basic fine-tune)
#   3. Raw base model
_pref_fused = MODELS_DIR / 'dpo' / 'fused'
_sft_fused  = FINETUNE_MODELS_DIR / 'round_0' / 'fused'

# Skip a fused model whose meta.json marks training as failed (NB19 writes
# this flag now). If meta.json is absent we still consider it — older runs
# don't have the flag.
def _meta_says_failed(model_dir):
    parent = Path(model_dir).parent
    meta = parent / 'meta.json'
    if not meta.exists():
        return False
    try:
        return bool(json.loads(meta.read_text()).get('stopped_early')) or \
               bool(json.loads(meta.read_text()).get('training_failed'))
    except Exception:
        return False

_candidates = []
if (_pref_fused / 'config.json').exists() and not _meta_says_failed(_pref_fused):
    _candidates.append(('NB19 preference-SFT', str(_pref_fused)))
if (_sft_fused / 'config.json').exists() and not _meta_says_failed(_sft_fused):
    _candidates.append(('NB18 SFT', str(_sft_fused)))

if _candidates:
    _label, BASE_MODEL = _candidates[0]
    print(f'Starting from {_label}: {BASE_MODEL}')
    if len(_candidates) > 1:
        print(f'  Fallback: {_candidates[1][0]} at {_candidates[1][1]}')
else:
    print(f'No fused fine-tune found, starting from raw base: {BASE_MODEL}')

NUM_ROUNDS          = 6      # how many improvement rounds to run
SAMPLES_PER_ROUND   = 600    # target — capped at len(DOMAINS)*N_TASKS_PER_DOMAIN below
MAX_REPAIR_ATTEMPTS = 5      # self-repair retries per sample
N_TASKS_PER_DOMAIN  = 3      # code_gen + debugging + translation per domain

# ── Hyperparams: match the improved NB18 config ────────────────────────────
# Previous defaults (LR 2e-5, rank 16/layers 8, batch 4 single-step) produced
# noisy training and overfitting. The settings below mirror what worked for
# NB18 (best val 0.174 after the rank-16→8 + weight-decay change).
ITERS_PER_ROUND     = 400
BATCH_SIZE          = 2      # was 4 — paired with grad-accum 8 = eff. batch 16
GRAD_ACCUM          = 4      # was 8 — smaller window reduces NaN risk on MoE
                             # (the 2026-05-02 round 2 NaN happened with GRAD_ACCUM=8)
LORA_RANK           = 8      # halved from 16 — limits memorisation
LORA_LAYERS         = 16     # raised from 8 — more capacity for diverse data
LEARNING_RATE       = 1e-5   # was 2e-5 — match NB18 stable rate
WEIGHT_DECAY        = 0.01   # AdamW weight decay via lora YAML config
MAX_SEQ_LEN         = 4096
STEPS_PER_EVAL      = 50     # was default 200 — gives 8 val checks per round
SAVE_EVERY          = 50     # save more checkpoints so best-ckpt selection has options

# Smoke mode: quick run for testing the pipeline end-to-end
SMOKE_MODE          = False
SMOKE_ROUNDS        = 2
SMOKE_SAMPLES       = 50

# Training stability guards (applied in run_training)
LOSS_EXPLODE_THRESHOLD = 8.0  # kill training if train_loss spikes above this
NO_IMPROVE_PATIENCE    = 4    # stop if val_loss doesn't beat best for this many evals
                              # (replaces "consecutive increases" — that one was
                              # fooled by val-loss noise on the 2026-04-30 NB18 run)

# Domains to draw from during generation
DOMAINS = [
    # ── Web APIs & CRUD ──────────────────────────────────────────────────────
    'todo list API with create, list, complete, and delete',
    'product catalog API with search and pagination',
    'blog posts API with comments and draft/publish workflow',
    'task manager API with priorities and due dates',
    'event registration API with capacity limits',
    'customer orders API with status state machine',
    'employee directory API with department filtering',
    'support ticket API with severity levels',
    'appointment scheduling API with conflict detection',
    'notification preferences API per user',
    'poll and voting API with results',
    'book library API with borrowing and returns',
    'user profiles API with avatar upload',
    'subscription billing API with renewal alerts',
    'project milestones API and task tracking',
    'contact list API with tags and search',
    'feature flag manager API with rollout percentage',
    'audit log viewer API with filtering',
    'API key management with rate limits',
    'webhook delivery API with retry logic',
    'document versioning API with diff',
    'team management API with roles',
    'coupon codes API with expiry and usage limits',
    'recipe collection API with ingredient search',
    'inventory tracking API with low-stock email alerts',

    # ── Real-Time & WebSocket ────────────────────────────────────────────────
    'real-time message board with WebSocket broadcast and in-memory storage',
    'live chat application with WebSocket rooms and user nicknames',
    'collaborative text editor with WebSocket sync',
    'real-time dashboard that pushes system metrics to connected clients',
    'stock ticker that streams price updates over WebSocket',
    'notification hub that broadcasts domain events to WebSocket subscribers',
    'live auction platform with real-time bidding over WebSocket',

    # ── File & Data Operations ───────────────────────────────────────────────
    'CSV file reader that parses rows and computes column statistics',
    'log file tail watcher that monitors a directory for new .log files',
    'JSON config merger that reads multiple config files and deep-merges them',
    'file deduplicator that hashes files in a directory and reports duplicates',
    'backup rotator that copies files to a dated archive folder and prunes old backups',
    'YAML validator that reads .yaml files from a directory and reports parse errors',
    'markdown link checker that reads .md files and validates internal links exist',
    'CSV to JSON converter that reads a CSV and writes each row as JSON',
    'file metadata reporter that lists files with size, modified date, and permissions',
    'directory tree printer that recursively lists a folder structure with indentation',
    'format-aware file reader that auto-detects JSON, YAML, and CSV formats',

    # ── Data Engineering & Pipelines ─────────────────────────────────────────
    'data pipeline that reads JSON records, filters by field value, and writes results',
    'ETL job that extracts data from an API, transforms fields, and stores in repository',
    'aggregation service that groups records by category and computes sum and average',
    'map-reduce pipeline that splits a dataset, processes chunks, and merges results',
    'data quality checker that validates records against a schema and emits error events',
    'time series aggregator that buckets events by hour and computes counts per bucket',
    'deduplication pipeline that reads records, detects duplicates by key, and keeps latest',
    'data enrichment service that fetches external API data and merges into local records',
    'batch processor that reads a file line by line, transforms each, and writes output',
    'streaming word counter that reads text files and computes word frequency by count',
    'stream processor for large files with O(1) memory and running statistics',
    'set operations pipeline with union, intersect, and difference on collections',

    # ── DevOps & Deployment ──────────────────────────────────────────────────
    'deployment status tracker with rollback state machine',
    'health check monitor that polls endpoints and emits alerts on failure',
    'service registry that tracks running services with heartbeat expiry',
    'release notes generator that reads git commit messages and formats a changelog',
    'environment config manager that reads env-specific YAML and validates required keys',
    'container restart watcher that monitors a socket for crash events and logs restarts',
    'canary deployment controller with traffic percentage and automatic rollback',
    'secret rotation scheduler that tracks expiry dates and emits renewal events',
    'incident tracker with severity levels and escalation state machine',
    'uptime dashboard that stores check results and computes availability percentage',

    # ── File System Watching & Automation ────────────────────────────────────
    'file watcher that monitors a directory and emits events on create, modify, and delete',
    'hot-reload config service that watches a YAML file and re-applies settings on change',
    'image thumbnail generator that watches an upload folder and creates resized copies',
    'log rotator that watches log file size and archives when threshold is exceeded',
    'sync service that watches a source directory and mirrors changes to a destination',
    'file classifier that watches an inbox folder and moves files to subfolders by extension',
    'template renderer that watches .mustache files and re-renders HTML on change',

    # ── CLI Tools & Utilities ────────────────────────────────────────────────
    'command-line calculator that parses arithmetic expressions from arguments',
    'password generator that produces random strings with configurable length and charset',
    'JSON pretty printer that reads stdin and writes formatted JSON',
    'base64 encoder/decoder CLI that converts files or stdin',
    'UUID generator that outputs UUIDs in different formats',
    'string diff tool that compares two text files and highlights differences',
    'CSV column extractor that selects specific columns by name',
    'hash checker that computes SHA256 of files and verifies against a manifest',
    'timestamp converter that translates between Unix epoch and human-readable dates',

    # ── Monitoring & Metrics ─────────────────────────────────────────────────
    'Prometheus metrics exporter that tracks request count, latency, and error rate',
    'disk usage monitor that checks free space and emits alerts above threshold',
    'process watchdog that monitors a PID and restarts the process on crash',
    'SLA tracker that computes uptime percentage from health check results',
    'request rate limiter that throttles API calls per client with sliding window',
    'system load monitor that parses uptime and serves metrics via JSON HTTP API',

    # ── Event-Driven & Messaging ─────────────────────────────────────────────
    'order fulfillment pipeline with state transitions from placed to shipped to delivered',
    'email notification hub that listens for domain events and sends templated emails',
    'chat message relay that receives messages and broadcasts to subscribers',
    'event sourcing ledger that stores events and rebuilds state from event history',
    'retry queue manager that re-emits failed events with exponential backoff',
    'multi-service orchestrator that coordinates startup and shutdown of multiple services',
    'repository observer that watches store/update/delete and logs all changes',

    # ── Domain-Specific Applications ─────────────────────────────────────────
    'weather dashboard that fetches external API data and displays forecasts',
    'expense tracker API with categories and monthly summaries',
    'fitness log API that records workouts and computes weekly statistics',
    'plant watering scheduler that tracks schedules and emits reminder events',
    'pet feeding tracker with meal logs and portion calculations',
    'home energy monitor that reads sensor data and computes daily usage',
    'meeting room booker API with time slot availability and conflict checks',
    'quiz engine API with questions, scoring, and leaderboard',

    # ── Social & Communication ───────────────────────────────────────────────
    'microblogging platform with posts, likes, and follower feeds',
    'Mastodon-style timeline reader with live streaming updates',
    'direct message service with conversation threads and read receipts',
    'forum API with topics, replies, and moderation actions',
    'comment system API with nested replies and vote counts',

    # ── E-Commerce & Finance ─────────────────────────────────────────────────
    'shopping cart API with add, remove, and checkout workflow',
    'payment processing service with transaction state machine',
    'price comparison aggregator that fetches from multiple sources',
    'invoice generator that computes line items, tax, and totals',
    'currency converter that fetches live rates from an external API',

    # ── Templates & Output ───────────────────────────────────────────────────
    'HTML report generator using Mustache-style templates',
    'context-aware formatter that adapts output for human, machine, and developer audiences',
    'email template engine with variable substitution and conditional sections',
    'markdown-to-HTML converter that processes .md files',

    # ── Plugins & Extensions ─────────────────────────────────────────────────
    'application with a Swift plugin for custom string transformations',
    'application with a C plugin for hashing and encoding',
    'application with a Python plugin for data analysis qualifiers',
    'application using plugin qualifiers for collection operations like shuffle and sort',

    # ── Networking & Sockets ─────────────────────────────────────────────────
    'TCP echo server that accepts connections and mirrors data back',
    'socket client that connects to a remote service and exchanges messages',
    'multi-service application with HTTP server and TCP socket listener',
    'HTTP client that fetches data from an external REST API and processes the response',
    'link header pagination client that follows paginated API responses',

    # ── Date, Time & Scheduling ──────────────────────────────────────────────
    'calendar event API with recurring events and date range queries',
    'cron-style job scheduler that executes tasks at configured intervals',
    'time tracking API with start/stop timers and daily summaries',
    'date range calculator for business days excluding holidays',

    # ── Security & Auth ──────────────────────────────────────────────────────
    'API gateway with token validation and request forwarding',
    'session manager that creates, validates, and expires auth tokens',
    'access control service with roles and permission checks',

    # ── Search & Indexing ────────────────────────────────────────────────────
    'full-text search API that indexes documents and returns ranked results',
    'tag-based search API with autocomplete suggestions',
    'log search service that filters log entries by level, date, and keyword',
]

# Cap target at the achievable per-round volume (each domain × each task type).
_max_unique = len(DOMAINS) * N_TASKS_PER_DOMAIN
if SAMPLES_PER_ROUND > _max_unique:
    print(f'⚠  SAMPLES_PER_ROUND ({SAMPLES_PER_ROUND}) > max unique '
          f'({len(DOMAINS)} domains × {N_TASKS_PER_DOMAIN} tasks = {_max_unique}).')
    print(f'   Capping to {_max_unique} so generation actually reaches the target.')
    SAMPLES_PER_ROUND = _max_unique


# ── Per-round LR decay (fix #2) ────────────────────────────────────────────
# Each successive round trains on top of an already-fine-tuned model. Keeping
# LR constant is a recipe for instability — we decay it geometrically.
def compute_round_lr(round_num, base_lr=LEARNING_RATE):
    # Halve every 2 rounds: round 0 = base, round 1 = base/1.5, round 2 = base/2...
    return base_lr / (1.0 + round_num * 0.5)

# ── Pass-rate-floor guard (fix #8: don't train hours for 0% pass rate) ─────
# After each round, if quick_eval reports below this threshold, increment a
# streak counter. Two consecutive low-quality rounds abort the loop early.
PASS_RATE_FLOOR        = 0.05   # 5% — below this means model isn't producing valid ARO
LOW_QUALITY_PATIENCE   = 2      # consecutive rounds at/below floor before abort

# ── Top-percentile sample cap (fix #4: drop largest gradients) ─────────────
# Even after the MAX_SEQ_LEN filter, the longest 5% of samples produce the
# largest single-step gradients on MoE models. Dropping them costs <5% of
# data but eliminates a major NaN trigger.
TOP_PERCENTILE_CAP     = 0.95   # keep samples whose token count is <= 95th percentile
print(f'Rounds:           {NUM_ROUNDS}')
print(f'Samples / round:  {SAMPLES_PER_ROUND}')
print(f'Repair attempts:  {MAX_REPAIR_ATTEMPTS}')
print(f'Iters / round:    {ITERS_PER_ROUND}')
print(f'Effective batch:  {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}  (was 16, halved for NaN robustness)')
print(f'Learning rate:    {LEARNING_RATE:.0e}')
print(f'Weight decay:     {WEIGHT_DECAY}')
print(f'LoRA rank/layers: {LORA_RANK} / {LORA_LAYERS}')
print(f'Max seq len:      {MAX_SEQ_LEN}')
print(f'Eval every:       {STEPS_PER_EVAL} iters')
print(f'Explosion guard:  >{LOSS_EXPLODE_THRESHOLD}')
print(f'Patience:         {NO_IMPROVE_PATIENCE} evals without new best val')
print(f'Pass-rate floor:  {PASS_RATE_FLOOR:.0%} (abort after {LOW_QUALITY_PATIENCE} rounds below)')
print(f'Top-pct cap:      keep <= {TOP_PERCENTILE_CAP:.0%}-ile token length')
print(f'Smoke mode:       {SMOKE_MODE}')
print(f'Domains:          {len(DOMAINS)}')
print(f'Base model:       {BASE_MODEL}')

## Shared helpers

In [ ]:
# Load knowledge base for system prompt
kb_path = DATA_ROOT / '02_knowledge' / 'knowledge.json'
if kb_path.exists():
    with open(kb_path) as f:
        kb = json.load(f)
    action_lines = [
        f'- {"/".join(a["verbs"][:3])}  (role: {a["role"]})'
        for a in kb['actions'] if a['verbs']
    ][:35]
else:
    kb = {}
    action_lines = []
    print('WARNING: knowledge.json not found — run notebooks 01+02 first.')

ARO_SYSTEM_PROMPT = f"""You are an expert ARO language programmer.
ARO (Action Result Object) is a DSL for expressing business logic as natural-language statements.

SYNTAX:
  (FeatureSetName: BusinessActivity) {{
      Verb [the] <result[:qualifier]> preposition [the] <object[:qualifier]>.
  }}

KEY RULES:
- String concatenation: ++ (NOT +)
- Variables: hyphenated lowercase e.g. <user-id>
- Forbidden prefixes: is-, with-, empty-
- HTTP path params: Extract the <id> from the <pathParameters: id>.
- Request body: Extract the <data> from the <request: body>.
- Exactly ONE Application-Start per application
- openapi.yaml required for HTTP server
- Return: <OK: status>, <Created: status>, <NotFound: status>

AVAILABLE ACTIONS:
{chr(10).join(action_lines)}

Generate complete, valid ARO that passes `aro check`. Wrap code in ```aro fences."""


def aro_check(code):
    # tempfile.tempdir is set to DATA_ROOT/tmp in the config cell
    try:
        with tempfile.TemporaryDirectory() as tmp:
            (Path(tmp) / 'main.aro').write_text(code)
            r = subprocess.run(['aro', 'check', tmp], capture_output=True, text=True, timeout=10)
            return r.returncode == 0, (r.stderr or r.stdout).strip()[:300]
    except FileNotFoundError:
        return None, 'aro_not_found'
    except subprocess.TimeoutExpired:
        return False, 'timeout'


def extract_aro_blocks(text):
    return [b.strip() for b in re.findall(r'```aro\n(.*?)```', text, re.DOTALL) if b.strip()]


def is_comment_heavy(code):
    """Return True if >50% of non-blank lines start with (* (comment)."""
    lines = [l.strip() for l in code.splitlines() if l.strip()]
    if not lines:
        return False
    comment_lines = sum(1 for l in lines if l.startswith('(*'))
    return comment_lines / len(lines) > 0.5


def get_few_shot_examples(kb, n=3):
    """Pick n random real examples from the knowledge base for few-shot prompting."""
    examples = kb.get('examples', [])
    if not examples:
        return ''
    picked = random.sample(examples, min(n, len(examples)))
    parts = ['Here are examples of valid ARO code for reference:\n']
    for ex in picked:
        name = ex.get('name', 'Example')
        files = ex.get('aro_files', {})
        if not files:
            continue
        # Use the first (or main) .aro file
        code = list(files.values())[0][:800]
        parts.append(f'### Example: {name}\n```aro\n{code}\n```\n')
    return '\n'.join(parts) if len(parts) > 1 else ''


def make_chat_fn(model, tokenizer, temperature=0.5):
    """Return a chat function bound to the given model."""
    sampler = make_sampler(temp=temperature)
    def chat(messages, max_tokens=1200):
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        return mlx_generate(model, tokenizer, prompt=prompt, max_tokens=max_tokens,
                            sampler=sampler, verbose=False)
    return chat


def generate_with_repair(chat_fn, instruction, max_tokens=1200):
    """Self-repair loop: generate -> aro check -> fix on failure -> repeat."""
    messages = [
        {'role': 'system', 'content': ARO_SYSTEM_PROMPT},
        {'role': 'user',   'content': instruction},
    ]
    for attempt in range(MAX_REPAIR_ATTEMPTS):
        output = chat_fn(messages, max_tokens=max_tokens)
        blocks = extract_aro_blocks(output)

        if not blocks:
            messages += [
                {'role': 'assistant', 'content': output},
                {'role': 'user',      'content': 'Wrap your ARO code in ```aro\n...\n```.'},
            ]
            continue

        passed, error = aro_check('\n\n'.join(blocks))
        if passed is True or passed is None:
            return output, attempt

        messages += [
            {'role': 'assistant', 'content': output},
            {'role': 'user',      'content': f'`aro check` error:\n\n```\n{error}\n```\n\nPlease fix the code.'},
        ]
    return None, MAX_REPAIR_ATTEMPTS

## Generation: produce one round of training data

In [ ]:
def run_generation_round(round_num, model, tokenizer, n_samples):
    """Generate n_samples for this round and save to data/rounds/round_{N}.jsonl.

    Each (domain, task_type) is sampled once per round — previously dedup was on
    domain alone, so the per-round cap was len(DOMAINS) regardless of target.
    Now each domain produces up to N_TASKS_PER_DOMAIN distinct samples (one
    code_generation + one debugging + one translation).

    Per-task temperature:
      - code_generation: 0.6 (creativity)
      - debugging:       0.3 (precision)
      - translation:     0.5 (balanced)

    Rejected samples (aro check failures + comment-heavy) are saved for DPO training.
    """
    output_file = ROUNDS_DIR / f'round_{round_num}.jsonl'
    reject_file = ROUNDS_DIR / f'round_{round_num}_rejects.jsonl'

    # Resume: track (domain, task_type) pairs already saved
    done = set()
    if output_file.exists():
        with open(output_file) as f:
            for line in f:
                try:
                    s = json.loads(line)
                    done.add((s.get('domain', ''), s.get('task_type', '')))
                except Exception:
                    pass
        print(f'  Resuming round {round_num}: {len(done)} already saved')

    few_shot_block = get_few_shot_examples(kb, n=3)

    stats = Counter()
    n_rejects = 0
    count = len(done)

    TASK_TEMPS = {
        'code_generation': 0.6,
        'debugging':       0.3,
        'translation':     0.5,
    }
    chat_fns = {
        task: make_chat_fn(model, tokenizer, temperature=temp)
        for task, temp in TASK_TEMPS.items()
    }

    # Build (domain, task_type) candidate list — proportionate to task mix.
    # 60/30/10 split across code_generation / debugging / translation.
    candidates = []
    for d in DOMAINS:
        candidates.append((d, 'code_generation'))   # always
        candidates.append((d, 'debugging'))         # always
        candidates.append((d, 'translation'))       # always (gives N_TASKS_PER_DOMAIN=3)
    random.shuffle(candidates)

    with open(output_file, 'a') as out_f, open(reject_file, 'a') as rej_f:
        for domain, task in candidates:
            if count >= n_samples:
                break
            key = (domain, task)
            if key in done:
                continue

            if task == 'code_generation':
                instruction = (
                    f'Write a complete ARO HTTP API for: {domain}.\n'
                    f'Include openapi.yaml and main.aro with Application-Start and all operationId handlers.\n\n'
                    f'{few_shot_block}'
                    f'## openapi.yaml\n```yaml\n...\n```\n\n## main.aro\n```aro\n...\n```'
                )
            elif task == 'debugging':
                ex_codes = [list(e['aro_files'].values())[0] for e in kb.get('examples', []) if e['aro_files']]
                if not ex_codes:
                    continue
                base_code = random.choice(ex_codes)[:1500]
                bugs = [
                    'Replace one `from` with `with` in a Retrieve statement',
                    'Replace `++` with `+` for string concatenation',
                    'Remove the Application-Start feature set',
                    'Use `is-valid` as a variable name (forbidden prefix)',
                    'Drop a closing `)` on a feature set definition',
                    'Use `+` instead of `++` between two strings',
                ]
                instruction = (
                    f'Here is valid ARO code:\n\n```aro\n{base_code}\n```\n\n'
                    f'Introduce bug: {random.choice(bugs)}\n\n'
                    f'{few_shot_block}'
                    f'Then: ## Buggy Code\n```aro\n...\n```\n\n## Fix\n```aro\n...\n```'
                )
            else:  # translation
                instruction = (
                    f'Write Python Flask pseudocode for: {domain}. '
                    f'Then translate it to ARO.\n\n'
                    f'{few_shot_block}'
                    f'## Python\n```python\n...\n```\n\n## ARO\n```aro\n...\n```'
                )

            chat_fn = chat_fns[task]
            output, attempts = generate_with_repair(chat_fn, instruction)
            stats[attempts] += 1

            if output:
                blocks = extract_aro_blocks(output)
                aro_code = '\n\n'.join(blocks) if blocks else ''
                if aro_code and is_comment_heavy(aro_code):
                    reject = {
                        'round': round_num, 'task_type': task,
                        'instruction': instruction[:200], 'output': output,
                        'domain': domain, 'reject_reason': 'comment_heavy',
                    }
                    rej_f.write(json.dumps(reject) + '\n')
                    rej_f.flush()
                    append_dropped(round_num, 'generation', task, domain,
                                   reason='comment_heavy',
                                   instruction=instruction, output=output)
                    n_rejects += 1
                    continue

                sample = {
                    'round': round_num, 'task_type': task,
                    'instruction': instruction[:200], 'output': output,
                    'domain': domain, 'repair_attempts': attempts,
                }
                out_f.write(json.dumps(sample) + '\n')
                out_f.flush()
                done.add(key)
                count += 1
                if count % 20 == 0:
                    print(f'  [{count}/{n_samples}] last attempts={attempts+1}, task={task}')
            else:
                reject = {
                    'round': round_num, 'task_type': task,
                    'instruction': instruction[:200], 'output': '(failed_all_repairs)',
                    'domain': domain, 'reject_reason': 'aro_check_failed',
                }
                rej_f.write(json.dumps(reject) + '\n')
                rej_f.flush()
                append_dropped(round_num, 'generation', task, domain,
                               reason='aro_check_failed',
                               instruction=instruction, output='(failed_all_repairs)')
                n_rejects += 1

    total = sum(stats.values())
    first_try_rate = 100 * stats.get(0, 0) // max(1, total)
    print(f'  Round {round_num} generation: {count} samples, {n_rejects} rejects, '
          f'first-try rate: {first_try_rate}%')
    return count

## Training: fine-tune on all rounds so far

In [ ]:
import math
import shutil
import glob
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from IPython import display as ipydisplay

plt.style.use('default')
plt.rcParams.update({
    'text.color': '#222222',
    'axes.labelcolor': '#222222',
    'xtick.color': '#333333',
    'ytick.color': '#333333',
    'axes.titlecolor': '#111111',
    'legend.edgecolor': '#cccccc',
    'legend.facecolor': 'white',
    'legend.framealpha': 1.0,
    'figure.facecolor': '#fafafa',
    'axes.facecolor': '#f9f9f9',
})


# ── Dropped-code logger ────────────────────────────────────────────────────
# Every sample that fails `aro check` (during generation, quick_eval, or smoke
# test) gets a row in run/<date>/21_dropped.csv so we can post-mortem what
# the model is doing wrong.
import csv as _csv_for_dropped
from datetime import datetime as _dt
from datetime import date as _date_for_dropped

_DROPPED_HEADER = ['timestamp', 'round', 'source', 'task_type', 'domain',
                   'reason', 'aro_error', 'instruction', 'output']

def append_dropped(round_num, source, task_type='', domain='',
                   reason='', aro_error='', instruction='', output=''):
    """Append one failing sample to run/<today>/21_dropped.csv."""
    run_dir = SCRIPT_DIR / 'run' / _date_for_dropped.today().isoformat()
    run_dir.mkdir(parents=True, exist_ok=True)
    csv_path = run_dir / '21_dropped.csv'
    write_header = not csv_path.exists()
    with open(csv_path, 'a', newline='') as f:
        w = _csv_for_dropped.writer(f)
        if write_header:
            w.writerow(_DROPPED_HEADER)
        w.writerow([
            _dt.now().isoformat(timespec='seconds'),
            round_num,
            source,
            task_type or '',
            (domain or '')[:120],
            reason or '',
            (aro_error or '')[:500],
            (instruction or '').replace('\n', ' ')[:300],
            (output or '')[:2000],
        ])

# ── Token-count helper for pre-flight filter ───────────────────────────────
_TOKENIZER_FOR_FILTER = None

def _set_token_filter_tokenizer(tok):
    global _TOKENIZER_FOR_FILTER
    _TOKENIZER_FOR_FILTER = tok


def _sample_token_count(rec):
    if _TOKENIZER_FOR_FILTER is None:
        return 0
    msgs = rec.get('messages')
    if msgs:
        text = _TOKENIZER_FOR_FILTER.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False)
    else:
        text = (rec.get('prompt', '') + rec.get('completion', '')) or rec.get('text', '')
    return len(_TOKENIZER_FOR_FILTER.encode(text, add_special_tokens=False))


def prepare_mlx_data(up_to_round, mlx_data_dir):
    """Combine all rounds up to up_to_round into mlx train/valid.jsonl.

    Two-stage filtering:
      1. Drop samples > MAX_SEQ_LEN tokens (mlx_lm would silently truncate them)
      2. Drop top (1 - TOP_PERCENTILE_CAP) by token length — these are the
         largest-gradient samples on MoE and primary NaN-inducers.
    """
    mlx_data_dir.mkdir(parents=True, exist_ok=True)
    all_samples = []

    initial_train = SCRIPT_DIR / '../data/05_dataset/mlx/train.jsonl'
    if initial_train.exists():
        with open(initial_train) as f:
            for line in f:
                if line.strip():
                    all_samples.append(json.loads(line))

    for r in range(up_to_round + 1):
        round_file = ROUNDS_DIR / f'round_{r}.jsonl'
        if not round_file.exists():
            continue
        with open(round_file) as f:
            for line in f:
                if not line.strip():
                    continue
                s = json.loads(line)
                output = s.get('output', '')
                instruction = s.get('instruction', '')
                if not output or not instruction:
                    continue
                if len(output.strip()) < 20:
                    continue
                import re as _re
                if _re.fullmatch(r'[!?.,:;\s]+', output.strip()):
                    continue
                all_samples.append({'messages': [
                    {'role': 'system',    'content': ARO_SYSTEM_PROMPT},
                    {'role': 'user',      'content': instruction},
                    {'role': 'assistant', 'content': output},
                ]})

    n_before = len(all_samples)

    # Stage 1: hard MAX_SEQ_LEN filter
    if _TOKENIZER_FOR_FILTER is not None:
        token_counts = [(s, _sample_token_count(s)) for s in all_samples]
        kept_with_tokens = [(s, n) for s, n in token_counts if n <= MAX_SEQ_LEN]
        n_truncated = len(token_counts) - len(kept_with_tokens)
        if n_truncated:
            longest = max(n for _, n in token_counts if n > MAX_SEQ_LEN)
            print(f'  Pre-flight: dropped {n_truncated} samples > {MAX_SEQ_LEN} '
                  f'tokens (longest {longest})')

        # Stage 2: top-percentile cap on remaining samples (drops the
        # extreme-gradient producers that aren't strictly truncated)
        if kept_with_tokens and 0 < TOP_PERCENTILE_CAP < 1:
            sorted_lens = sorted(n for _, n in kept_with_tokens)
            cap_idx = max(1, int(len(sorted_lens) * TOP_PERCENTILE_CAP)) - 1
            cap_tokens = sorted_lens[cap_idx]
            before = len(kept_with_tokens)
            kept_with_tokens = [(s, n) for s, n in kept_with_tokens if n <= cap_tokens]
            n_pct_dropped = before - len(kept_with_tokens)
            if n_pct_dropped:
                print(f'  Pre-flight: dropped {n_pct_dropped} samples above '
                      f'{int(TOP_PERCENTILE_CAP*100)}th percentile ({cap_tokens} tokens)')

        all_samples = [s for s, _ in kept_with_tokens]

    random.seed(42)
    random.shuffle(all_samples)
    split = int(len(all_samples) * 0.95)
    train, valid = all_samples[:split], all_samples[split:]

    def write_jsonl(data, path):
        with open(path, 'w') as f:
            for item in data:
                f.write(json.dumps(item) + '\n')

    write_jsonl(train, mlx_data_dir / 'train.jsonl')
    write_jsonl(valid, mlx_data_dir / 'valid.jsonl')
    print(f'  MLX data: {len(train)} train, {len(valid)} valid '
          f'(rounds 0-{up_to_round}, {n_before}->{len(all_samples)} after filter)')
    return len(all_samples), len(train), len(valid)


# ── Live loss plot using display_id (replaces clear_output) ────────────────
_LOSS_FIG_HANDLE = None

def _draw_loss_plot(ax, fig, train_iters, train_losses, val_iters, val_losses, round_num):
    global _LOSS_FIG_HANDLE
    ax.clear()
    if train_losses:
        ax.plot(train_iters, train_losses, color='steelblue', linewidth=1.5, label='train loss')
    if val_losses:
        ax.plot(val_iters, val_losses, color='tomato', linewidth=2,
                marker='o', markersize=5, label='val loss')
    ax.set_title(f'Round {round_num} - training loss', fontsize=13)
    ax.set_xlabel('iteration')
    ax.set_ylabel('loss')
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    if _LOSS_FIG_HANDLE is None:
        _LOSS_FIG_HANDLE = ipydisplay.display(fig, display_id=f'loss_round_{round_num}')
    else:
        _LOSS_FIG_HANDLE.update(fig)


def _reset_loss_handle():
    global _LOSS_FIG_HANDLE
    _LOSS_FIG_HANDLE = None


def _is_nan(s):
    try:
        return math.isnan(float(s))
    except (TypeError, ValueError):
        return s.strip().lower() == 'nan'


def _select_best_checkpoint(adapter_dir, val_iters, val_losses, total_iters):
    ckpts = sorted(glob.glob(str(adapter_dir / '*_adapters.safetensors')))
    final_adapter = adapter_dir / 'adapters.safetensors'
    if not ckpts or not val_iters:
        return
    best_val = min(val_losses)
    best_val_iter = val_iters[val_losses.index(best_val)]

    parsed = []
    for cf in ckpts:
        m = re.match(r'^(\d+)_adapters\.safetensors$', Path(cf).name)
        if m:
            parsed.append((int(m.group(1)), cf))
    if not parsed:
        return
    parsed.sort(key=lambda x: x[0])
    best_ckpt_iter, best_ckpt_path = min(parsed, key=lambda x: abs(x[0] - best_val_iter))

    last_iter = parsed[-1][0]
    if final_adapter.exists() and best_ckpt_iter == last_iter \
            and abs(total_iters - best_val_iter) <= abs(best_ckpt_iter - best_val_iter):
        print(f'  Best checkpoint: final adapter (iter {best_val_iter}, val {best_val:.4f})')
        return
    shutil.copy2(best_ckpt_path, final_adapter)
    print(f'  Best checkpoint: iter {best_ckpt_iter} (val {best_val:.4f}) '
          f'-> promoted to adapters.safetensors')


def run_training(round_num, data_dir, base_model_path, n_train, n_valid,
                 resume_adapter_file=None, learning_rate=None):
    """Run mlx_lm lora for this round.

    Args:
      base_model_path: stable BASE_MODEL — does NOT change between rounds.
      resume_adapter_file: previous round's adapter file, or None for round 0.
      learning_rate: per-round LR (decayed via compute_round_lr).

    Returns: (adapter_dir, train_losses, val_losses, status). status is one of
    'ok', 'nan', 'explosion'. The caller decides whether to fuse or skip.
    """
    adapter_dir = ITERATIVE_MODELS_DIR / f'round_{round_num}' / 'adapter'
    adapter_dir.mkdir(parents=True, exist_ok=True)
    log_path = adapter_dir / 'train.log'

    if learning_rate is None:
        learning_rate = LEARNING_RATE

    effective_batch = max(1, min(BATCH_SIZE, n_train, n_valid))
    val_batches     = max(1, min(25, n_valid // max(1, effective_batch)))

    import yaml
    lora_config_path = adapter_dir / 'lora_config.yaml'
    lora_config = {
        'lora_parameters': {
            'rank': LORA_RANK,
            'dropout': 0.0,
            'scale': 20.0,
        },
        'optimizer': 'adamw',
        'optimizer_config': {'adamw': {'weight_decay': float(WEIGHT_DECAY)}},
    }
    with open(lora_config_path, 'w') as f:
        yaml.safe_dump(lora_config, f)

    cmd = [
        sys.executable, '-m', 'mlx_lm', 'lora',
        '-c',                        str(lora_config_path),
        '--model',                   str(base_model_path),
        '--data',                    str(data_dir),
        '--train',
        '--batch-size',              str(effective_batch),
        '--grad-accumulation-steps', str(GRAD_ACCUM),
        '--num-layers',              str(LORA_LAYERS),
        '--learning-rate',           str(learning_rate),
        '--iters',                   str(ITERS_PER_ROUND),
        '--save-every',              str(SAVE_EVERY),
        '--steps-per-eval',          str(STEPS_PER_EVAL),
        '--val-batches',             str(val_batches),
        '--adapter-path',            str(adapter_dir),
        '--max-seq-length',          str(MAX_SEQ_LEN),
        '--mask-prompt',
    ]

    if resume_adapter_file is not None and Path(resume_adapter_file).exists():
        cmd += ['--resume-adapter-file', str(resume_adapter_file)]
        print(f'  Resuming from adapter: {resume_adapter_file}')

    print(f'  Training round {round_num} '
          f'(eff_batch={effective_batch}x{GRAD_ACCUM}, val_batches={val_batches}, '
          f'lr={learning_rate:.2e}, wd={WEIGHT_DECAY}, '
          f'rank={LORA_RANK}, layers={LORA_LAYERS}, '
          f'eval_every={STEPS_PER_EVAL}, max_seq={MAX_SEQ_LEN})')
    print(f'  Log: {log_path}')

    re_train = re.compile(r'Iter\s+(\d+):\s+Train loss\s+([-\w.]+)', re.IGNORECASE)
    re_val   = re.compile(r'Iter\s+(\d+):\s+Val loss\s+([-\w.]+)',   re.IGNORECASE)

    train_iters, train_losses = [], []
    val_iters,   val_losses   = [], []
    all_lines = []

    _best_val_seen = float('inf')
    _no_improve_n  = 0
    _stopped_early = False
    _loss_exploded = False
    _loss_nan      = False

    _reset_loss_handle()
    fig, ax = plt.subplots(figsize=(8, 4))

    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )

    update_every = 5
    last_draw    = 0

    with open(log_path, 'w') as log_f:
        for line in proc.stdout:
            line = line.rstrip()
            all_lines.append(line)
            log_f.write(line + '\n')
            log_f.flush()

            m = re_train.search(line)
            if m:
                it       = int(m.group(1))
                loss_str = m.group(2)
                if _is_nan(loss_str):
                    _loss_nan = True
                    print(f'\n  [NaN] train loss at iter {it} - aborting round.')
                    proc.terminate()
                    break
                loss = float(loss_str)
                train_iters.append(it)
                train_losses.append(loss)

                if loss > LOSS_EXPLODE_THRESHOLD:
                    _loss_exploded = True
                    print(f'\n  LOSS EXPLOSION at iter {it}: {loss:.3f} > {LOSS_EXPLODE_THRESHOLD}')
                    proc.terminate()
                    break

                if len(train_iters) - last_draw >= update_every:
                    _draw_loss_plot(ax, fig, train_iters, train_losses,
                                    val_iters, val_losses, round_num)
                    last_draw = len(train_iters)
                continue

            m = re_val.search(line)
            if m:
                it       = int(m.group(1))
                loss_str = m.group(2)
                if _is_nan(loss_str):
                    _loss_nan = True
                    print(f'\n  [NaN] val loss at iter {it} - aborting round.')
                    proc.terminate()
                    break
                loss = float(loss_str)
                val_iters.append(it)
                val_losses.append(loss)
                _draw_loss_plot(ax, fig, train_iters, train_losses,
                                val_iters, val_losses, round_num)
                last_draw = len(train_iters)

                if loss < _best_val_seen:
                    _best_val_seen = loss
                    _no_improve_n  = 0
                else:
                    _no_improve_n += 1
                    print(f'  no improvement vs best ({_best_val_seen:.4f}) '
                          f'- streak {_no_improve_n}/{NO_IMPROVE_PATIENCE}')
                    if _no_improve_n >= NO_IMPROVE_PATIENCE:
                        _stopped_early = True
                        print(f'\n  EARLY STOP round {round_num} at iter {it}')
                        proc.terminate()
                        break

    proc.wait()

    if train_losses or val_losses:
        _draw_loss_plot(ax, fig, train_iters, train_losses,
                        val_iters, val_losses, round_num)
    plt.close(fig)

    if _loss_nan:
        print(f'  Round {round_num} aborted due to NaN.')
        return adapter_dir, train_losses, val_losses, 'nan'
    if _loss_exploded:
        print(f'  Round {round_num} aborted due to loss explosion.')
        return adapter_dir, train_losses, val_losses, 'explosion'
    if proc.returncode not in (0, -15) and not _stopped_early:
        tail = all_lines[-40:] if len(all_lines) >= 40 else all_lines
        print(f'\n--- training failed (last {len(tail)} lines) ---')
        print('\n'.join(tail))
        raise RuntimeError(f'Training failed at round {round_num} (exit {proc.returncode})')

    if train_losses:
        print(f'  Train loss: {train_losses[0]:.4f} -> {train_losses[-1]:.4f}  '
              f'(best {min(train_losses):.4f})')
    if val_losses:
        best_v = min(val_losses)
        best_vi = val_iters[val_losses.index(best_v)]
        print(f'  Val   loss: {val_losses[0]:.4f} -> {val_losses[-1]:.4f}  '
              f'(best {best_v:.4f} @ iter {best_vi})')

    _select_best_checkpoint(adapter_dir, val_iters, val_losses, ITERS_PER_ROUND)

    return adapter_dir, train_losses, val_losses, 'ok'


def fuse_model(round_num, base_model_path, adapter_dir):
    fused_dir = ITERATIVE_MODELS_DIR / f'round_{round_num}' / 'fused'
    cmd = [
        sys.executable, '-m', 'mlx_lm', 'fuse',
        '--model',        str(base_model_path),
        '--adapter-path', str(adapter_dir),
        '--save-path',    str(fused_dir),
    ]
    print(f'  Fusing round {round_num}...')
    result = subprocess.run(cmd)
    if result.returncode != 0:
        raise RuntimeError(f'Fuse failed at round {round_num}')
    return fused_dir


## Evaluation helper (syntax pass rate)

In [ ]:
def _aro_check_with_contract(openapi_yaml, main_aro):
    """Run aro check against a directory containing both files. Returns (ok, msg)."""
    try:
        with tempfile.TemporaryDirectory() as tmp:
            d = Path(tmp)
            if openapi_yaml:
                (d / 'openapi.yaml').write_text(openapi_yaml)
            (d / 'main.aro').write_text(main_aro)
            r = subprocess.run(['aro', 'check', str(d)],
                               capture_output=True, text=True, timeout=15)
            return r.returncode == 0, (r.stderr or r.stdout).strip()[:300]
    except FileNotFoundError:
        return None, 'aro_not_found'
    except subprocess.TimeoutExpired:
        return False, 'timeout'


def _extract_openapi_and_aro(text):
    """Extract openapi.yaml + main.aro blocks from a generated response.

    Models trained on the NB21 prompt produce:
      ## openapi.yaml
      ```yaml
      ...
      ```
      ## main.aro
      ```aro
      ...
      ```
    Returns (openapi_yaml_or_None, main_aro_or_None).
    """
    yaml_blocks = re.findall(r'```yaml\s*\n(.*?)```', text, re.DOTALL)
    aro_blocks  = re.findall(r'```aro\s*\n(.*?)```',  text, re.DOTALL)
    openapi = next((b.strip() for b in yaml_blocks if 'openapi' in b.lower()), None)
    main_aro = '\n\n'.join(b.strip() for b in aro_blocks) if aro_blocks else None
    return openapi, main_aro


def quick_eval(model, tokenizer, n=60):
    """Generate n samples using the SAME prompt format as training, write
    openapi.yaml + main.aro to a tmpdir, and run `aro check` on the directory.

    Previous bug: training used "Write a complete ARO HTTP API" (multi-file),
    eval used "Write a short ARO feature set" (single-file) and ran aro check
    on the extracted ARO standalone. That always failed because the ARO
    references types defined in openapi.yaml. Result: 0/60 even when the
    model was producing perfectly valid output.
    """
    chat_fn = make_chat_fn(model, tokenizer, temperature=0.4)
    passed = 0
    sample_domains = random.sample(DOMAINS, min(n, len(DOMAINS)))
    if n > len(DOMAINS):
        sample_domains = (DOMAINS * (n // len(DOMAINS) + 1))[:n]
        random.shuffle(sample_domains)

    failures_by_reason = Counter()
    for domain in sample_domains:
        instruction = (
            f'Write a complete ARO HTTP API for: {domain}.\n'
            f'Include openapi.yaml and main.aro with Application-Start and all operationId handlers.'
        )
        messages = [
            {'role': 'system', 'content': ARO_SYSTEM_PROMPT},
            {'role': 'user',   'content': instruction},
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        output = mlx_generate(model, tokenizer, prompt=prompt, max_tokens=1200, verbose=False)

        openapi, main_aro = _extract_openapi_and_aro(output)
        if not main_aro:
            failures_by_reason['no_aro_block'] += 1
            append_dropped(getattr(append_dropped, '_eval_round', -1),
                           'quick_eval', task_type='code_generation',
                           domain=domain, reason='no_aro_block',
                           instruction=instruction, output=output)
            continue
        ok, _msg = _aro_check_with_contract(openapi or '', main_aro)
        if ok is True:
            passed += 1
        elif ok is None:
            failures_by_reason['aro_cli_missing'] += 1
            append_dropped(getattr(append_dropped, '_eval_round', -1),
                           'quick_eval', task_type='code_generation',
                           domain=domain, reason='aro_cli_missing',
                           aro_error=_msg, instruction=instruction, output=main_aro)
        else:
            failures_by_reason['check_failed'] += 1
            append_dropped(getattr(append_dropped, '_eval_round', -1),
                           'quick_eval', task_type='code_generation',
                           domain=domain, reason='check_failed',
                           aro_error=_msg, instruction=instruction, output=main_aro)

    rate = passed / n
    breakdown = ', '.join(f'{k}={v}' for k, v in failures_by_reason.most_common()) or 'none'
    print(f'  Syntax pass rate: {passed}/{n} = {rate:.0%}  (failures: {breakdown})')
    return rate


def validate_eval_pipeline():
    """Run 3 hand-written valid ARO programs through the same extract+check
    pipeline. If any of them fails, the eval pipeline itself is broken — abort
    the loop so we don't burn hours measuring against a wrong yardstick."""

    case1_main = (
        '(Application-Start: Hello) {\n'
        '    Log "hello" to the <console>.\n'
        '    Return an <OK: status> for the <startup>.\n'
        '}\n'
    )
    case2_openapi = (
        'openapi: 3.0.3\n'
        'info:\n'
        '  title: Test API\n'
        '  version: 1.0.0\n'
        'paths:\n'
        '  /users/{id}:\n'
        '    get:\n'
        '      operationId: getUser\n'
        '      parameters:\n'
        '        - name: id\n'
        '          in: path\n'
        '          required: true\n'
        '          schema:\n'
        '            type: string\n'
        '      responses:\n'
        '        200:\n'
        '          description: ok\n'
    )
    case2_main = (
        '(getUser: User API) {\n'
        '    Extract the <id> from the <pathParameters: id>.\n'
        '    Return an <OK: status> with <id>.\n'
        '}\n'
        '\n'
        '(Application-Start: User Service) {\n'
        '    Start the <http-server> with <contract>.\n'
        '    Keepalive the <application> for the <events>.\n'
        '    Return an <OK: status> for the <startup>.\n'
        '}\n'
    )
    # Use the canonical Create+Compute pattern (matches Examples/Calculator).
    # `<5>` and `<2>` were rejected by aro check because integer literals
    # cannot appear as identifiers inside angle brackets — only letters,
    # digits, and hyphens. Real ARO uses Create-then-Compute.
    case3_main = (
        '(Application-Start: Compute) {\n'
        '    Create the <a> with 5.\n'
        '    Create the <b> with 2.\n'
        '    Compute the <doubled> from <a> * <b>.\n'
        '    Log <doubled> to the <console>.\n'
        '    Return an <OK: status> for the <run>.\n'
        '}\n'
    )

    cases = [
        ('hello world', None,           case1_main),
        ('user lookup', case2_openapi,  case2_main),
        ('compute',     None,           case3_main),
    ]
    print('  Validating eval pipeline against 3 hand-written examples...')
    failures = []
    for name, openapi, main_aro in cases:
        ok, msg = _aro_check_with_contract(openapi or '', main_aro)
        if ok is True:
            print(f'    [OK]  {name}')
        else:
            failures.append((name, msg))
            print(f'    [FAIL] {name}: {msg[:200]}')
    if failures:
        raise RuntimeError(
            f'Eval pipeline broken — {len(failures)}/3 hand-written examples '
            f'failed aro check. Fix _aro_check_with_contract or the aro CLI '
            f'before running rounds.'
        )
    print('  Eval pipeline OK (3/3 hand-written examples pass)')


## Run the loop

In [ ]:
round_metrics = []
current_model_path = BASE_MODEL   # starts as MODEL_ID (via config.py), becomes local path after round 0

# Evaluate base model before any training
print('=== Baseline evaluation ===')
print(f'Loading {current_model_path} ...')
model, tokenizer = load(str(current_model_path))
baseline_rate = quick_eval(model, tokenizer)
round_metrics.append({'round': -1, 'model': str(current_model_path), 'syntax_pass_rate': baseline_rate})
del model   # free VRAM between generation and training

In [ ]:
import hashlib

if SMOKE_MODE:
    NUM_ROUNDS = SMOKE_ROUNDS
    SAMPLES_PER_ROUND = SMOKE_SAMPLES
    print(f'SMOKE MODE: {SMOKE_ROUNDS} rounds, {SMOKE_SAMPLES} samples each')


def model_fingerprint(model_dir):
    """Hash a sample of bytes from start, middle, and end of every shard."""
    md = Path(str(model_dir))
    shards = sorted(md.glob('model*.safetensors')) or sorted(md.glob('*.safetensors'))
    if not shards:
        return 'n/a'
    h = hashlib.md5()
    for s in shards:
        size = s.stat().st_size
        with open(s, 'rb') as f:
            h.update(f.read(4096))
            f.seek(size // 2)
            h.update(f.read(4096))
            f.seek(max(0, size - 4096))
            h.update(f.read(4096))
        h.update(s.name.encode())
    return h.hexdigest()[:12]


def validate_model(model, tokenizer, n_probes=3):
    probes = random.sample(DOMAINS, min(n_probes, len(DOMAINS)))
    ok = 0
    for d in probes:
        msg = [
            {'role': 'system', 'content': ARO_SYSTEM_PROMPT},
            {'role': 'user',   'content': f'Write a tiny ARO feature set for {d}.'},
        ]
        p = tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
        out = mlx_generate(model, tokenizer, prompt=p, max_tokens=120, verbose=False)
        if (out and any(ch.isalpha() for ch in out)
                and not out.startswith('!!!!')
                and ('(' in out or '<' in out)):
            ok += 1
    print(f'  Validation: {ok}/{n_probes} probes produced usable output')
    return ok >= max(1, n_probes // 2)


# Self-check the eval pipeline before any training
print('\n=== Eval pipeline self-check ===')
validate_eval_pipeline()


# Anchor BASE_MODEL for training; chain adapters between rounds.
# Previously each round trained on top of the *fused* model from the prior
# round, which compounds float16 quantization drift. Now training always uses
# the original BASE_MODEL and resumes from the prior round's adapter. Fusing
# still happens for downstream tools (NB23 distillation, NB24 packaging),
# but training never reads from a fused output.
TRAINING_BASE_MODEL    = BASE_MODEL          # fixed, set in cell 4
prev_adapter_file      = None                # round 0 starts fresh
prev_adapter_dir       = None
generation_model_path  = BASE_MODEL          # for round 0 generation
generation_adapter_dir = None

_weight_hashes = {}
_low_quality_streak = 0
_loop_aborted = False

for round_num in range(NUM_ROUNDS):
    t_start = time.time()
    print(f'\n{"="*60}')
    print(f'ROUND {round_num}')
    print(f'{"="*60}')

    # --- 1. Load model for generation: BASE + latest adapter ---
    print(f'\n[1/6] Loading generation model: BASE + {generation_adapter_dir or "(no adapter)"}')
    if generation_adapter_dir is not None:
        model, tokenizer = load(str(TRAINING_BASE_MODEL),
                                 adapter_path=str(generation_adapter_dir))
    else:
        model, tokenizer = load(str(generation_model_path))

    if round_num == 0 and str(generation_model_path) != str(MODEL_ID):
        print(f'  Validating fine-tuned base model before adopting...')
        if not validate_model(model, tokenizer):
            print(f'  [WARN] Base model failed validation -> falling back to {MODEL_ID}')
            del model, tokenizer
            generation_model_path = MODEL_ID
            TRAINING_BASE_MODEL   = MODEL_ID
            model, tokenizer = load(str(generation_model_path))

    _set_token_filter_tokenizer(tokenizer)

    fp_target = generation_adapter_dir if generation_adapter_dir else TRAINING_BASE_MODEL
    _weight_hash = model_fingerprint(fp_target)
    print(f'  Weight hash: {_weight_hash}  (source: {Path(str(fp_target)).name})')
    _weight_hashes[round_num] = _weight_hash

    # --- 2. Generate ---
    print(f'\n[2/6] Generating {SAMPLES_PER_ROUND} samples...')
    t_gen = time.time()
    n_generated = run_generation_round(round_num, model, tokenizer, SAMPLES_PER_ROUND)
    gen_min = (time.time() - t_gen) / 60
    print(f'  Generation done: {n_generated} samples in {gen_min:.1f} min '
          f'({n_generated / max(gen_min, 0.1):.0f} samples/min)')

    del model

    # --- 3. Prepare combined MLX dataset ---
    print(f'\n[3/6] Preparing dataset (rounds 0-{round_num})...')
    mlx_data_dir = ROUNDS_DIR / f'mlx_round_{round_num}'
    total_samples, n_train, n_valid = prepare_mlx_data(round_num, mlx_data_dir)
    print(f'  Cumulative dataset: {total_samples:,} total ({n_train:,} train / {n_valid:,} valid)')

    del tokenizer
    _set_token_filter_tokenizer(None)

    # --- 4. Fine-tune (resumes from prior adapter, decayed LR) ---
    round_lr = compute_round_lr(round_num)
    print(f'\n[4/6] Fine-tuning (iters={ITERS_PER_ROUND}, lr={round_lr:.2e}, '
          f'rank={LORA_RANK}, layers={LORA_LAYERS}, max_seq={MAX_SEQ_LEN})')
    t_train = time.time()
    adapter_dir, train_losses, val_losses, train_status = run_training(
        round_num, mlx_data_dir, TRAINING_BASE_MODEL, n_train, n_valid,
        resume_adapter_file=prev_adapter_file,
        learning_rate=round_lr,
    )
    train_min = (time.time() - t_train) / 60
    print(f'  Training done in {train_min:.1f} min, status={train_status}')

    if train_status in ('nan', 'explosion'):
        print(f'  [WARN] Round {round_num} failed ({train_status}). '
              f'Keeping previous adapter; next round will halve LR further.')
        _low_quality_streak += 1
        round_metrics.append({
            'round':            round_num,
            'model':            str(TRAINING_BASE_MODEL),
            'n_generated':      n_generated,
            'total_dataset':    total_samples,
            'syntax_pass_rate': 0.0,
            'minutes':          round((time.time() - t_start) / 60, 1),
            'gen_minutes':      round(gen_min, 1),
            'train_minutes':    round(train_min, 1),
            'weight_hash':      _weight_hashes.get(round_num, 'n/a'),
            'lr':               round_lr,
            'train_status':     train_status,
        })
        if _low_quality_streak >= LOW_QUALITY_PATIENCE:
            print(f'\n[ABORT] {LOW_QUALITY_PATIENCE} failed rounds in a row.')
            _loop_aborted = True
            break
        with open(MODELS_DIR / 'loop_metrics.json', 'w') as f:
            json.dump(round_metrics, f, indent=2)
        continue

    # --- 5. Fuse for downstream tools ---
    print(f'\n[5/6] Fusing adapter into base model (for NB23/NB24)...')
    t_fuse = time.time()
    fused_dir = fuse_model(round_num, TRAINING_BASE_MODEL, adapter_dir)
    fuse_sec = time.time() - t_fuse
    print(f'  Fused in {fuse_sec:.0f}s -> {fused_dir}')

    # Smoke test on base+adapter
    print(f'\n  Smoke test (5 samples)...')
    _smoke_model, _smoke_tok = load(str(TRAINING_BASE_MODEL), adapter_path=str(adapter_dir))
    _smoke_pass = 0
    for _sd in random.sample(DOMAINS, 5):
        _sm = [{'role': 'system', 'content': ARO_SYSTEM_PROMPT},
               {'role': 'user',   'content': f'Write a short ARO feature set for {_sd}.'}]
        _sp = _smoke_tok.apply_chat_template(_sm, tokenize=False, add_generation_prompt=True)
        _so = mlx_generate(_smoke_model, _smoke_tok, prompt=_sp, max_tokens=600, verbose=False)
        _sb = extract_aro_blocks(_so)
        if _sb:
            _ok, _smsg = aro_check('\n\n'.join(_sb))
            if _ok is True:
                _smoke_pass += 1
            else:
                append_dropped(round_num, 'smoke_test',
                               task_type='code_generation',
                               domain=_sd, reason='check_failed',
                               aro_error=str(_smsg or ''),
                               instruction=f'Write a short ARO feature set for {_sd}.',
                               output='\n\n'.join(_sb))
        else:
            append_dropped(round_num, 'smoke_test',
                           task_type='code_generation',
                           domain=_sd, reason='no_aro_block',
                           instruction=f'Write a short ARO feature set for {_sd}.',
                           output=_so)
    print(f'  Smoke test: {_smoke_pass}/5 pass')
    if _smoke_pass == 0:
        print(f'  [WARN] 0/5 smoke test samples pass aro check!')
    del _smoke_model, _smoke_tok

    # --- 6. Evaluate ---
    print(f'\n[6/6] Evaluating round {round_num} model (n=60)...')
    model, tokenizer = load(str(TRAINING_BASE_MODEL), adapter_path=str(adapter_dir))
    append_dropped._eval_round = round_num   # tag so quick_eval rows know the round
    pass_rate = quick_eval(model, tokenizer, n=60)
    del model

    elapsed = (time.time() - t_start) / 60
    prev_rate = round_metrics[-1]['syntax_pass_rate'] if round_metrics else 0
    delta = pass_rate - prev_rate

    round_metrics.append({
        'round':            round_num,
        'model':            str(fused_dir),
        'n_generated':      n_generated,
        'total_dataset':    total_samples,
        'syntax_pass_rate': pass_rate,
        'minutes':          round(elapsed, 1),
        'gen_minutes':      round(gen_min, 1),
        'train_minutes':    round(train_min, 1),
        'weight_hash':      _weight_hashes.get(round_num, 'n/a'),
        'lr':               round_lr,
        'train_status':     train_status,
    })

    print(f'\n{"~"*60}')
    print(f'  Round {round_num} complete in {elapsed:.0f} min')
    print(f'    Generated:    {n_generated} samples ({gen_min:.1f} min)')
    print(f'    Dataset:      {total_samples:,} cumulative')
    print(f'    Training:     {train_min:.1f} min  lr={round_lr:.2e}')
    print(f'    Pass rate:    {pass_rate:.0%}  ({"+" if delta >= 0 else ""}{delta:.0%} vs previous)')
    print(f'    Weight hash:  {_weight_hashes.get(round_num, "n/a")}')
    print(f'{"~"*60}')

    # Low-quality guard
    if pass_rate <= PASS_RATE_FLOOR:
        _low_quality_streak += 1
        print(f'  [LOW-QUALITY] pass rate {pass_rate:.0%} <= floor {PASS_RATE_FLOOR:.0%} '
              f'(streak {_low_quality_streak}/{LOW_QUALITY_PATIENCE})')
        if _low_quality_streak >= LOW_QUALITY_PATIENCE:
            print(f'\n[ABORT] {LOW_QUALITY_PATIENCE} consecutive rounds at or below '
                  f'{PASS_RATE_FLOOR:.0%} pass rate. Stopping the loop.')
            print(f'        Training is producing nothing useful. Investigate the eval')
            print(f'        prompt format and the generated training data quality before')
            print(f'        running more rounds.')
            _loop_aborted = True
            break
    else:
        _low_quality_streak = 0

    # Chain adapters for the next round
    prev_adapter_file      = adapter_dir / 'adapters.safetensors'
    prev_adapter_dir       = adapter_dir
    generation_adapter_dir = adapter_dir
    current_model_path     = fused_dir   # legacy var some downstream code uses

    with open(MODELS_DIR / 'loop_metrics.json', 'w') as f:
        json.dump(round_metrics, f, indent=2)

if _loop_aborted:
    with open(MODELS_DIR / 'loop_metrics.json', 'w') as f:
        json.dump(round_metrics, f, indent=2)
    print(f'\n=== LOOP ABORTED — see metrics above ===')


## Results

In [ ]:
print('\n=== Self-Improvement Summary ===\n')
print(f'{"Round":<8} {"Pass Rate":<12} {"Dataset":<12} {"Minutes":<10} {"Model"}')
print('─' * 70)
for m in round_metrics:
    r     = m['round']
    label = 'baseline' if r == -1 else str(r)
    rate  = f'{m["syntax_pass_rate"]:.0%}'
    ds    = str(m.get('total_dataset', '—'))
    mins  = str(m.get('minutes', '—'))
    mod   = Path(m['model']).name if Path(m['model']).exists() else m['model'].split('/')[-1]
    print(f'{label:<8} {rate:<12} {ds:<12} {mins:<10} {mod}')

print(f'\nFinal model: {current_model_path}')
print(f'Metrics:     {ITERATIVE_MODELS_DIR / "loop_metrics.json"}')

In [ ]:
# ── Self-improvement graph ──────────────────────────────────────────────
import csv as _csv
from datetime import date as _date

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

rounds  = [m['round'] for m in round_metrics]
labels  = ['base' if r == -1 else f'R{r}' for r in rounds]
rates   = [m['syntax_pass_rate'] for m in round_metrics]

# --- Panel 1: Syntax pass rate ---
ax = axes[0]
colors = ['#888888' if r == -1 else 'steelblue' for r in rounds]
bars = ax.bar(labels, [r * 100 for r in rates], color=colors, edgecolor='white', width=0.6)
for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f'{rate:.0%}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylabel('pass rate (%)')
ax.set_title('Syntax pass rate per round', fontsize=13)
ax.set_ylim(0, max(r * 100 for r in rates) * 1.25 + 5)
ax.grid(axis='y', alpha=0.3)

# --- Panel 2: Cumulative dataset size ---
ax = axes[1]
trained = [m for m in round_metrics if m['round'] >= 0]
if trained:
    t_labels = [f'R{m["round"]}' for m in trained]
    datasets = [m.get('total_dataset', 0) for m in trained]
    ax.bar(t_labels, datasets, color='mediumseagreen', edgecolor='white', width=0.6)
    for i, (lbl, ds) in enumerate(zip(t_labels, datasets)):
        ax.text(i, ds + max(datasets) * 0.02, f'{ds:,}', ha='center', va='bottom', fontsize=10)
    ax.set_ylabel('samples')
    ax.set_title('Cumulative dataset size', fontsize=13)
    ax.grid(axis='y', alpha=0.3)
else:
    ax.text(0.5, 0.5, 'no training rounds', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Cumulative dataset size', fontsize=13)

# --- Panel 3: Time breakdown ---
ax = axes[2]
if trained:
    gen_mins   = [m.get('gen_minutes', 0) for m in trained]
    train_mins = [m.get('train_minutes', 0) for m in trained]
    x = range(len(trained))
    ax.bar(t_labels, gen_mins, color='sandybrown', edgecolor='white', width=0.6, label='generation')
    ax.bar(t_labels, train_mins, bottom=gen_mins, color='cornflowerblue',
           edgecolor='white', width=0.6, label='training')
    ax.set_ylabel('minutes')
    ax.set_title('Time per round', fontsize=13)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
else:
    ax.text(0.5, 0.5, 'no training rounds', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Time per round', fontsize=13)

fig.suptitle('NB21 — Iterative Self-Improvement', fontsize=15, fontweight='bold', y=1.02)
fig.tight_layout()
_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(MODELS_DIR / 'loop_summary.png', dpi=150, bbox_inches='tight', facecolor='#fafafa')
plt.savefig(_run_dir / '20_iterative_loop.png', dpi=150, bbox_inches='tight', facecolor='#fafafa')
plt.show()
print(f'Saved: {MODELS_DIR / "loop_summary.png"}')
print(f'Saved: {_run_dir / "20_iterative_loop.png"}')

# ── CSV export ──────────────────────────────────────────────────────────
_csv_path = _run_dir / '20_iterative_loop.csv'
with open(_csv_path, 'w', newline='') as _cf:
    _writer = _csv.writer(_cf)
    _writer.writerow(['round', 'pass_rate', 'dataset_size', 'weight_hash', 'training_minutes'])
    for m in round_metrics:
        _writer.writerow([
            'baseline' if m['round'] == -1 else m['round'],
            f'{m["syntax_pass_rate"]:.4f}',
            m.get('total_dataset', ''),
            m.get('weight_hash', ''),
            m.get('train_minutes', ''),
        ])
print(f'Saved: {_csv_path}')

In [ ]:
# ── Focused summary: rounds vs success rate ────────────────────────────
# Saves a clean, standalone graphic highlighting how the syntax pass rate
# evolves across self-improvement rounds. Useful for reports / READMEs.

fig, ax = plt.subplots(figsize=(10, 6))

rounds_all = [m['round'] for m in round_metrics]
labels_all = ['base' if r == -1 else f'Round {r}' for r in rounds_all]
rates_pct  = [m['syntax_pass_rate'] * 100 for m in round_metrics]

x = list(range(len(rounds_all)))
baseline_pct = rates_pct[0] if rates_pct else 0.0

# Baseline reference line
ax.axhline(baseline_pct, color='#bbbbbb', linestyle='--', linewidth=1,
           label=f'baseline ({baseline_pct:.0f}%)', zorder=1)

# Line + markers showing progression
ax.plot(x, rates_pct, color='#2b6cb0', linewidth=2.5, zorder=2)
point_colors = ['#888888' if r == -1 else '#2b6cb0' for r in rounds_all]
ax.scatter(x, rates_pct, s=140, c=point_colors, edgecolors='white',
           linewidths=2, zorder=3)

# Value labels + per-round delta vs previous
for i, (xi, rate) in enumerate(zip(x, rates_pct)):
    ax.text(xi, rate + 2.2, f'{rate:.0f}%', ha='center', va='bottom',
            fontsize=12, fontweight='bold', color='#111111')
    if i > 0:
        delta = rate - rates_pct[i - 1]
        sign  = '+' if delta >= 0 else ''
        color = '#2f855a' if delta >= 0 else '#c53030'
        ax.text(xi, rate - 4.5, f'{sign}{delta:.0f} pp', ha='center', va='top',
                fontsize=10, color=color, fontweight='bold')

# Best round highlight
if len(rates_pct) > 1:
    best_i = max(range(len(rates_pct)), key=lambda i: rates_pct[i])
    ax.scatter([best_i], [rates_pct[best_i]], s=260, facecolors='none',
               edgecolors='#d69e2e', linewidths=2.5, zorder=4,
               label=f'best: {labels_all[best_i]} ({rates_pct[best_i]:.0f}%)')

ax.set_xticks(x)
ax.set_xticklabels(labels_all)
ax.set_ylabel('syntax pass rate (%)', fontsize=12)
ax.set_xlabel('self-improvement round', fontsize=12)
ax.set_ylim(0, max(max(rates_pct) * 1.25 + 8, 25))
ax.set_title('ARO-Coder — success rate across self-improvement rounds',
             fontsize=14, fontweight='bold', pad=14)
ax.grid(axis='y', alpha=0.3)
ax.legend(loc='lower right', fontsize=10)

# Subtitle with total lift
if len(rates_pct) > 1:
    lift = rates_pct[-1] - rates_pct[0]
    sign = '+' if lift >= 0 else ''
    fig.text(0.5, -0.02,
             f'{len(rates_pct) - 1} rounds  •  total lift {sign}{lift:.0f} pp  '
             f'(baseline {baseline_pct:.0f}% → final {rates_pct[-1]:.0f}%)',
             ha='center', fontsize=11, color='#555555', style='italic')

fig.tight_layout()

summary_path = ITERATIVE_MODELS_DIR / 'rounds_success_rate.png'
summary_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(summary_path, dpi=160, bbox_inches='tight', facecolor='#fafafa')
_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(_run_dir / '20_iterative_loop_rates.png', dpi=160, bbox_inches='tight', facecolor='#fafafa')
plt.show()
print(f'Saved: {summary_path}')
print(f'Saved: {_run_dir / "20_iterative_loop_rates.png"}')

## Summary

In [ ]:
print('=' * 65)
print('notebook 21 — ITERATIVE SELF-IMPROVEMENT SUMMARY')
print('=' * 65)

print(f'\nConfiguration:')
print(f'  Rounds:          {NUM_ROUNDS}')
print(f'  Samples/round:   {SAMPLES_PER_ROUND}')
print(f'  Repair attempts: {MAX_REPAIR_ATTEMPTS}')
print(f'  Iters/round:     {ITERS_PER_ROUND}')
print(f'  Learning rate:   {LEARNING_RATE:.0e}')
print(f'  Weight decay:    {WEIGHT_DECAY}')
print(f'  Effective batch: {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}')
print(f'  LoRA rank/layers:{LORA_RANK} / {LORA_LAYERS}')
print(f'  Eval every:      {STEPS_PER_EVAL} iters')
print(f'  Patience:        {NO_IMPROVE_PATIENCE} evals without new best')
print(f'  Smoke mode:      {SMOKE_MODE}')

if round_metrics:
    baseline = next((m for m in round_metrics if m['round'] == -1), None)
    final    = round_metrics[-1]

    print(f'\nProgress:')
    for m in round_metrics:
        r     = m['round']
        label = 'baseline' if r == -1 else f'round {r}'
        rate  = f'{m["syntax_pass_rate"]:.0%}'
        ds    = f'{m.get("total_dataset","?"):,}' if isinstance(m.get("total_dataset"), int) else '—'
        mins  = f'{m.get("minutes","?"):.0f}m' if isinstance(m.get("minutes"), (int, float)) else '—'
        whash = m.get('weight_hash', '')
        print(f'  {label:<12}  pass_rate={rate:<6}  dataset={ds:<8}  time={mins:<6}  hash={whash}')

    if baseline and len(round_metrics) > 1:
        start_rate = baseline['syntax_pass_rate']
        end_rate   = final['syntax_pass_rate']
        delta      = end_rate - start_rate
        print(f'\nImprovement: {start_rate:.0%} → {end_rate:.0%}  (Δ {delta:+.0%})')
        if end_rate < 0.10:
            print('  ⚠  Syntax pass rate still very low (<10%). Model is not learning ARO.')
            print('     Check data quality in NB17 and that aro check works on samples.')
        elif end_rate < 0.30:
            print('  ⚠  Modest improvement but still low. Run more rounds or increase SAMPLES_PER_ROUND.')
        else:
            print('  ✓  Meaningful syntax pass rate improvement.')

    _hashes = [m.get('weight_hash', '') for m in round_metrics if m['round'] >= 0 and m.get('weight_hash')]
    if len(_hashes) >= 2 and len(set(_hashes)) == 1:
        print('\n  ⚠  WARNING: All rounds have identical weight hashes!')
        print('     Weights are NOT changing between rounds — check fuse step.')
else:
    print('\n  (no round metrics recorded)')

print(f'\nFinal model: {current_model_path}')
print(f'Metrics log: {MODELS_DIR / "loop_metrics.json"}')

print(f'\nNext steps:')
print(f'  1. Run NB20 (evaluation) on final model to get full metric breakdown.')
print(f'  2. If pass_rate < 20%, re-run NB17 to rebuild dataset with better balance.')
print(f'  3. If loss exploded in any round, LEARNING_RATE is still too high — try {LEARNING_RATE/2:.0e}.')
print('=' * 65)